# 18 · Contextual Compression

Trim retrieved chunks to the spans that actually answer the query.

**Analogy handbook:** [compression](../retriever-analogy-handbook.html#compression)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


Retrieve full chunks
       ↓
LLM checks each chunk against query
       ↓
Extract only query-relevant content
       ↓
Return compressed documents

### Learning: CONTEXTUAL COMPRESSION RETRIEVER PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `CONTEXTUAL COMPRESSION RETRIEVER PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# CONTEXTUAL COMPRESSION RETRIEVER PRACTICAL
# LangChain + LLMChainExtractor
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------------------

# %pip install -U langchain-classic langchain-openai


# ============================================================
# 2. IMPORTS
# ============================================================

from langchain_openai import ChatOpenAI

from langchain_classic.retrievers import (
    ContextualCompressionRetriever
)

from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor
)


### Learning: CREATE BASE RETRIEVER

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE BASE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 3. CREATE BASE RETRIEVER
# ============================================================

# We use the existing Chroma vector_store.
#
# First, normal vector retrieval will fetch documents.
# After that, contextual compression will be applied.

base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Base retriever created successfully.")

Base retriever created successfully.


### Learning: CREATE LLM

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created successfully.")

LLM created successfully.


### Learning: CREATE LLM-BASED DOCUMENT COMPRESSOR

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `CREATE LLM-BASED DOCUMENT COMPRESSOR` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:

# ============================================================
# 5. CREATE LLM-BASED DOCUMENT COMPRESSOR
# ============================================================

# LLMChainExtractor extracts only the portions of
# retrieved documents that are relevant to the query.

compressor = LLMChainExtractor.from_llm(
    llm
)

print("LLMChainExtractor created successfully.")


LLMChainExtractor created successfully.


### Learning: CREATE CONTEXTUAL COMPRESSION RETRIEVER

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `CREATE CONTEXTUAL COMPRESSION RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 6. CREATE CONTEXTUAL COMPRESSION RETRIEVER
# ============================================================

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

print(
    "ContextualCompressionRetriever "
    "created successfully."
)


ContextualCompressionRetriever created successfully.


### Learning: USER QUERY

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: USER QUERY.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. USER QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

print("\nUSER QUERY:")
print(query)


# ============================================================
# 8. NORMAL RETRIEVAL
# ============================================================

normal_documents = base_retriever.invoke(
    query
)


print(
    "\nNORMAL RETRIEVAL RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nDOCUMENT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Original length:",
        len(document.page_content)
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )



USER QUERY:
How does Llama 2 improve safety?

NORMAL RETRIEVAL RESULTS

DOCUMENT 1
Page: 3
Section: introduction
Original length: 274
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

DOCUMENT 2
Page: 3
Section: introduction
Original length: 978
----------------------------------------------------------------------------------------------------
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-sour

### Learning: RUN CONTEXTUAL COMPRESSION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RUN CONTEXTUAL COMPRESSION.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. RUN CONTEXTUAL COMPRESSION
# ============================================================

compressed_documents = (
    compression_retriever.invoke(
        query
    )
)


print(
    "\n\nCONTEXTUAL COMPRESSION RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    compressed_documents,
    start=1
):

    print(
        f"\nCOMPRESSED DOCUMENT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Compressed length:",
        len(document.page_content)
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )




CONTEXTUAL COMPRESSION RESULTS

COMPRESSED DOCUMENT 1
Page: 3
Section: introduction
Compressed length: 104
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.

COMPRESSED DOCUMENT 2
Page: 3
Section: introduction
Compressed length: 530
----------------------------------------------------------------------------------------------------
Llama 2-Chat models generally perform better than existing open-source models. They also appear to be on par with some of the closed-source models, at least on the human evaluations we performed (see Figures 1 and 3). We have taken measures to increase the safety of these models, using safety-specific data annotation and tuning, as well as conducting red-teaming and employing iterative evaluations. Additionally, this paper contributes a thorough description of our fine-tuning methodology and approach to

### Learning: COMPARE ORIGINAL VS COMPRESSED

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `COMPARE ORIGINAL VS COMPRESSED` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 10. COMPARE ORIGINAL VS COMPRESSED
# ============================================================

print(
    "\n\nORIGINAL VS COMPRESSED"
)

print(
    "=" * 100
)


print(
    "\nOriginal documents returned:",
    len(normal_documents)
)

print(
    "Compressed documents returned:",
    len(compressed_documents)
)


original_characters = sum(
    len(document.page_content)
    for document in normal_documents
)


compressed_characters = sum(
    len(document.page_content)
    for document in compressed_documents
)


print(
    "\nTotal original characters:",
    original_characters
)

print(
    "Total compressed characters:",
    compressed_characters
)




ORIGINAL VS COMPRESSED

Original documents returned: 4
Compressed documents returned: 4

Total original characters: 3093
Total compressed characters: 1521


### Learning: CALCULATE COMPRESSION REDUCTION

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `CALCULATE COMPRESSION REDUCTION` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 11. CALCULATE COMPRESSION REDUCTION
# ============================================================

if original_characters > 0:

    reduction_percentage = (
        (
            original_characters
            -
            compressed_characters
        )
        /
        original_characters
    ) * 100

    print(
        f"\nContext reduction: "
        f"{reduction_percentage:.2f}%"
    )



Context reduction: 50.82%


### Learning: SIDE-BY-SIDE COMPARISON

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `SIDE-BY-SIDE COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 12. SIDE-BY-SIDE COMPARISON
# ============================================================

print(
    "\n\nSIDE-BY-SIDE CONCEPTUAL COMPARISON"
)

print(
    "=" * 100
)


for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nORIGINAL DOCUMENT {i}"
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:1000]
    )


print(
    "\n\nAFTER CONTEXTUAL COMPRESSION"
)

print(
    "=" * 100
)


for i, document in enumerate(
    compressed_documents,
    start=1
):

    print(
        f"\nCOMPRESSED DOCUMENT {i}"
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )




SIDE-BY-SIDE CONCEPTUAL COMPARISON

ORIGINAL DOCUMENT 1
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

ORIGINAL DOCUMENT 2
----------------------------------------------------------------------------------------------------
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we pe

### Learning: CREATE REUSABLE FUNCTION

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Defines helper logic for: CREATE REUSABLE FUNCTION.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:

# ============================================================
# 13. CREATE REUSABLE FUNCTION
# ============================================================

def contextual_compression_search(
    query: str
):

    documents = (
        compression_retriever.invoke(
            query
        )
    )

    return documents


### Learning: TEST REUSABLE FUNCTION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEST REUSABLE FUNCTION` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:

# ============================================================
# 14. TEST REUSABLE FUNCTION
# ============================================================

query = (
    "What role did human feedback play "
    "in training Llama 2-Chat?"
)


results = contextual_compression_search(
    query
)


print(
    "\n\nREUSABLE CONTEXTUAL COMPRESSION SEARCH"
)

print(
    "=" * 100
)


for i, document in enumerate(
    results,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "\nRelevant extracted content:"
    )

    print(
        document.page_content
    )

    print(
        "\n" + "-" * 100
    )



REUSABLE CONTEXTUAL COMPRESSION SEARCH

RESULT 1
Page: 5
Section: pretraining

Relevant extracted content:
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain parallel with model enhancements is crucial to ensure the reward models remain within distribution.

----------------------------------------------------------------------------------------------------

RESULT 2
Page: 18
Section: fine_tuning

Relevant extracted content:
3.4.2 Human Evaluation
Human evaluation is often considered the gold standard for judging models for natura

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 15. FINAL CONCEPTUAL FLOW
# ============================================================

"""
CONTEXTUAL COMPRESSION RETRIEVAL

User Query
      ↓
Base Retriever
      ↓
Retrieve Top-K Documents
      ↓
LLMChainExtractor
      ↓
Query + Document 1
      ↓
Extract relevant content

Query + Document 2
      ↓
Extract relevant content

Query + Document 3
      ↓
Extract relevant content

Query + Document 4
      ↓
Extract relevant content
      ↓
Remove irrelevant information
      ↓
Compressed Documents
      ↓
LLM / RAG Pipeline
"""


print(
    "\nContextual Compression Retriever "
    "practical completed successfully."
)


Contextual Compression Retriever practical completed successfully.


### Learning: CREATE LLM

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created successfully.")